##### Import statements:

In [201]:
import os
import pathlib
import inspect
import functools
import pickle
import numpy as np
import pandas as pd
import json
import socket
import multiprocessing as mp
import functools
import itertools
import torch
import matplotlib.pyplot as plt

hostname = socket.gethostname()

if 'rc.zi.columbia.edu' in hostname:
    from ws.general import find_df_constants, matches_template, class_def2str
    from ws.simulate_task import load_sim_params, load_task_def, simulate_session
    from ws.miscellaneous_sparseauto import mdl_geometry_pipeline, fmt_ae_metadata, generate_hparams_df, causal_mask
    from ws.plot import plot_iterate_autoencoder_results, plot_ccgps_by_layer, plot_pars_by_layer
    base = os.path.join('/', 'mnt', 'smb', 'locker', 'issa-locker', 'users', 'Dan', 'code', 'ws') 
else:
    from simulation_whiskers.general import find_df_constants, matches_template, class_def2str
    from simulation_whiskers.simulate_task import load_sim_params, load_task_def, simulate_session, causal_mask
    from simulation_whiskers.miscellaneous_sparseauto import mdl_geometry_pipeline, fmt_ae_metadata, generate_hparams_df, causal_mask
    #from simulation_whiskers.plot import plot_iterate_autoencoder_results, plot_autoencoder_geometry
    from simulation_whiskers.plot import plot_iterate_autoencoder_results, plot_ccgps_by_layer, plot_pars_by_layer
    base = os.path.join('C:\\', 'Users', 'danie' 'Documents', 'code_libraries', 'simulation_whiskers')

from analysis_metadata.analysis_metadata import Metadata, increment_dir_name, write_metadata
import time

##### Define parameters:

In [223]:
# Define task definitions:
task_defs = [
    
    # Task 0:
    [
     functools.partial(matches_template, template={'freq_sh' : 2}), 
     functools.partial(matches_template, template={'freq_sh' : 15})
     ],
    
    # Task 1:
    [
     functools.partial(matches_template, template={'time_mov' : 10}),
     functools.partial(matches_template, template={'time_mov' : 17})
     ]
    ]

# Define general variables:
n_files = 10
n_geo_subsamples = 1
sum_inpt=False
xor=True
zscore_data = False
sig_init = 1.0
save_learning = False
chunked_reconstruction_loss = False

# Define simulation parameters:
concavity = [0]
n_whisk = 2
prob_poiss = 1.01
noise_w = 0.3
spread = 'auto'
speed = 2.0
ini_phase_m = 0
ini_phase_spr = 100
delay_time = 0
freq_m = 3.0
freq_std = 0.1
std_reset = 0
t_total = 2
dt = 0.1
dx = 0.01
n_trials_pre = 50
amp = 2
freq_sh = [2, 15]
z1 = [4]
max_rad = 50
n_rad = 4
disp = 4.5
theta = [0]
steps_mov = [10, 17]
rad_vec = [6]
init_position = 0

# Entangler model parameters:
tngl_n_hidden = 80
tngl_beta_rec = 1.0
tngl_beta_pr = 1.0
tngl_beta_sp = 0
tngl_p_norm = 1
tngl_n_epochs = 50
tngl_batch_size = 64
tngl_lr = 0.001
tngl_sig_init = 1 
tngl_sig_neu = 0.1 

# Model parameters:
mdl_type = 'prediction'
n_hidden = 20
sig_init = 1 
sig_neu = 0.1 
lr = 0.001
beta0 = 0
beta1 = 0
beta_rec = 0
beta_xor = 0
n_epochs = 50
batch_size = 10
beta_sp = 0
beta_pr = 0
p_norm = 2
n_splits = 5
n_predictor_bins = 10
n_predicted_bins = 4
n_offsets = 1

# Compute parameters:
gpu = False
n_cores = 1

# Do some custom, ad-hoc hyperparameter selection:
#beta_lins=10**np.arange(0, 5, 1)
beta_lins= [10**2.5]
#beta_lins = np.array([0] + list(beta_lins))
#beta_lins = [0, 10**2.5, 10**5]
#n_hiddens = [40, 240]
n_hiddens = [40]
hparams = [{'beta_rec':x[0], 'n_hidden':x[1]} for x in list(itertools.product(beta_lins, n_hiddens))]
#hparams = None

# Output directory:
if 'rc.zi.columbia' in hostname:
    base_output_directory = os.path.join(base, 'results')
else:
    base_output_directory='E:\\simulation_whiskers\\results\\'
run_base_name='run'
sv=True

##### Deal with some preliminaries:

In [203]:
simulation_cols = ['concavity', 'n_whisk', 'prob_poiss', 'noise_w', 'spread',
     'speed', 'ini_phase_m', 'ini_phase_spr', 'delay_time', 'freq_m', 'freq_std',
     'std_reset', 't_total', 'dt', 'dx', 'n_trials_pre', 'n_files', 'amp', 'freq_sh',
     'z1', 'max_rad', 'n_rad', 'disp', 'theta', 'steps_mov', 'rad_vec', 'init_position']

autoencoder_cols = ['mdl_type', 'n_hidden', 'sig_init', 'sig_neu', 'lr', 'beta0',
    'beta1', 'beta_rec', 'beta_xor', 'n_epochs', 'batch_size', 'beta_sp', 'p_norm',
    'beta_pr', 'n_splits', 'n_predictor_bins', 'n_predicted_bins', 'n_offsets']

# TODO: A lot of this seems really inefficient; streamline this somehow?

# Put simulation params into dict:
sim_params = {
    'n_whisk' : n_whisk,
    'prob_poiss' : prob_poiss,
    'noise_w' : noise_w,
    'spread' : spread,
    'speed' : speed,
    'ini_phase_m' : ini_phase_m,
    'ini_phase_spr' : ini_phase_spr, 
    'delay_time' : delay_time, 
    'freq_m' : freq_m, 
    'freq_std' : freq_std,
    'std_reset' : std_reset,
    't_total' : t_total,
    'dt' : dt,
    'dx' : dx,
    'n_trials_pre' : n_trials_pre, 
    'amp' : amp,
    'freq_sh' : freq_sh,
    'z1' : z1,
    'max_rad' : max_rad,
    'n_rad' : n_rad,
    'disp' : disp,
    'theta' : theta,
    'steps_mov' : steps_mov,
    'rad_vec' : rad_vec,
    'init_position' : init_position,
}

n_feat = n_whisk*2

##### Simulate initial whisker data used to fit entangler model:

In [204]:
# Simulate train and test sessions: 
tngl_train_session=simulate_session(sim_params, sum_bins=False)
tngl_train_session['split'] = 'train'
tngl_train_session['trial_num'] = np.arange(tngl_train_session.shape[0])

tngl_test_session=simulate_session(sim_params, sum_bins=False)
tngl_test_session['split'] = 'test'
tngl_test_session['trial_num'] = np.arange(tngl_test_session.shape[0])

# Merge train and test:
tngl_sim_df = pd.concat([tngl_train_session, tngl_test_session], axis=0)

# Unrwap features:
tngl_sim_df['features'] = tngl_sim_df.apply(lambda x : np.reshape(x.features,-1), axis=1)

# Split into predicted and predictor features:
tngl_sim_df = causal_mask(tngl_sim_df, n_feat, n_predictor_bins, n_predicted_bins, n_offsets)
tngl_sim_df['predicted_features'] = tngl_sim_df['predictor_features'] # Predicted and predictor features will be same for entangling autoencoder

##### Fit entangler model:

In [290]:
tngl_mdl_params = {
    'mdl_type' : "autoencoder",
    'n_hidden' : tngl_n_hidden,
    'beta_rec' : tngl_beta_rec,
    'beta_pr' : tngl_beta_pr,    
    'beta_sp' : tngl_beta_sp,    
    'n_epochs' : tngl_n_epochs, 
    'batch_size' : tngl_batch_size,
    'lr' : tngl_lr, 
    'p_norm' : tngl_p_norm, 
    'sig_init' : tngl_sig_init,
    'sig_neu' :  tngl_sig_neu,
    'beta0' : 0,
    'beta1' : 0,
    'beta_xor' : 0,
}

tngl_results = mdl_geometry_pipeline(sim_params, task_defs, tngl_mdl_params, sessions_in=tngl_sim_df, save_learning=False, sum_inpt=False)
entangler = tngl_results['mdl']

Fitting autoencoder...
0 rec  0.16235531866550446 ce  0.0 sp  0.14807508885860443 total  -16.491594418883324
49 rec  0.07180257886648178 ce  0.0 sp  0.08329512178897858 total  -66.42147592455149
fit_autoencoder duration=1.7276544570922852


In [222]:
tngl_results.keys()

dict_keys(['ae_df', 'perf_df', 'geo_df', 'train_sessions', 'test_sessions', 'mdl'])

##### Generate dataframe of hyperparameters for main model:

In [207]:
hparams_df = generate_hparams_df(hparams=hparams, task_defs=task_defs, n_files=n_files, 
     xor=xor, n_geo_subsamples=n_geo_subsamples, zscore_data=zscore_data, 
     save_perf=False, sum_inpt=sum_inpt, chunked_reconstruction_loss=False, 
     save_learning=save_learning, gpu=gpu, save_sessions=False, verbose=False, 
     concavity=concavity, n_whisk=n_whisk, prob_poiss=prob_poiss, noise_w=noise_w, 
     spread=spread, speed=speed, ini_phase_m=ini_phase_m, ini_phase_spr=ini_phase_spr, 
     delay_time=delay_time, freq_m=freq_m, freq_std=freq_std, std_reset=std_reset, 
     t_total=t_total, dt=dt, dx=dx, n_trials_pre=n_trials_pre, n_repeats=n_files, 
     amp=amp, freq_sh=freq_sh, z1=z1, max_rad=max_rad, n_rad=n_rad, disp=disp, 
     theta=theta, steps_mov=steps_mov, rad_vec=rad_vec, init_position=init_position, 
     mdl_type=mdl_type, n_hidden=n_hidden, sig_init=sig_init, sig_neu=sig_neu, 
     lr=lr, beta0=beta0, beta1=beta1, beta_rec=beta_rec, beta_xor=beta_xor, 
     beta_sp=beta_sp, beta_pr=beta_pr, n_epochs=n_epochs, batch_size=batch_size, 
     p_norm=p_norm, n_splits=n_splits, n_predictor_bins=n_predictor_bins, 
     n_predicted_bins=n_predicted_bins, n_offsets=n_offsets)

# Verify parameters before executing:
hparam_strs = list(hparams_df.apply(lambda x : 'model={}, n_hidden={}, beta_rec={}, beta_sp={}, beta_pr={}, n_epochs={}'.format(x.mdl_type,x.n_hidden, x.beta_rec, x.beta_sp, x.beta_pr, x.n_epochs), axis=1))
print('Running following hyperparameters:\n')
print('\n'.join(hparam_strs))
yn = input('\nProceed? (y/n)')
if '__file__' not in dir():
    if yn == 'y':
        pass
    else: 
        raise AssertionError('User aborted execution.')

Running following hyperparameters:

model=prediction, n_hidden=40, beta_rec=316.22776601683796, beta_sp=0, beta_pr=0, n_epochs=50
model=prediction, n_hidden=40, beta_rec=316.22776601683796, beta_sp=0, beta_pr=0, n_epochs=50
model=prediction, n_hidden=40, beta_rec=316.22776601683796, beta_sp=0, beta_pr=0, n_epochs=50
model=prediction, n_hidden=40, beta_rec=316.22776601683796, beta_sp=0, beta_pr=0, n_epochs=50
model=prediction, n_hidden=40, beta_rec=316.22776601683796, beta_sp=0, beta_pr=0, n_epochs=50
model=prediction, n_hidden=40, beta_rec=316.22776601683796, beta_sp=0, beta_pr=0, n_epochs=50
model=prediction, n_hidden=40, beta_rec=316.22776601683796, beta_sp=0, beta_pr=0, n_epochs=50
model=prediction, n_hidden=40, beta_rec=316.22776601683796, beta_sp=0, beta_pr=0, n_epochs=50
model=prediction, n_hidden=40, beta_rec=316.22776601683796, beta_sp=0, beta_pr=0, n_epochs=50
model=prediction, n_hidden=40, beta_rec=316.22776601683796, beta_sp=0, beta_pr=0, n_epochs=50



Proceed? (y/n) y


##### Generate simulated trials for each set of hyperparameters:

In [291]:
# Generate simulated contacts:
sim_df = pd.DataFrame()
for hidx, hparams in hparams_df.iterrows():

    # Test:
    curr_train_sim = simulate_session(sim_params, sum_bins=False)
    curr_train_sim['split'] = 'train'
    curr_train_sim['trial_num'] = np.arange(curr_train_sim.shape[0])

    # Train:
    curr_test_sim = simulate_session(sim_params, sum_bins=False)
    curr_test_sim['split'] = 'test'
    curr_test_sim['trial_num'] = np.arange(curr_train_sim.shape[0])

    # Merge:
    curr_sim = pd.concat([curr_train_sim, curr_test_sim], axis=0)

    # Unrwap features:
    curr_sim['features'] = curr_sim.apply(lambda x : np.reshape(x.features,-1), axis=1)
    
    # Split simulated whisker data into predicted and predictor features:
    curr_sim = causal_mask(curr_sim, n_feat, n_predictor_bins, n_predicted_bins, n_offsets)
    curr_sim['sim_idx'] = hidx
    
    # Concatenate across repeats:
    sim_df = pd.concat([sim_df, curr_sim], axis=0)

# Pass predictor features through entangler model:
sim_df['predictor_features'] = sim_df.apply(lambda x : entangler.enc(torch.Tensor(x.predictor_features.astype(np.float32))).detach().numpy(), axis=1)

df.shape=(400, 13)
predictor_feat_df.shape=(400, 4)
predicted_feat_df.shape=(400, 4)
df_masked.shape=(400, 16)
df.shape=(400, 13)
predictor_feat_df.shape=(400, 4)
predicted_feat_df.shape=(400, 4)
df_masked.shape=(400, 16)
df.shape=(400, 13)
predictor_feat_df.shape=(400, 4)
predicted_feat_df.shape=(400, 4)
df_masked.shape=(400, 16)
df.shape=(400, 13)
predictor_feat_df.shape=(400, 4)
predicted_feat_df.shape=(400, 4)
df_masked.shape=(400, 16)
df.shape=(400, 13)
predictor_feat_df.shape=(400, 4)
predicted_feat_df.shape=(400, 4)
df_masked.shape=(400, 16)
df.shape=(400, 13)
predictor_feat_df.shape=(400, 4)
predicted_feat_df.shape=(400, 4)
df_masked.shape=(400, 16)
df.shape=(400, 13)
predictor_feat_df.shape=(400, 4)
predicted_feat_df.shape=(400, 4)
df_masked.shape=(400, 16)
df.shape=(400, 13)
predictor_feat_df.shape=(400, 4)
predicted_feat_df.shape=(400, 4)
df_masked.shape=(400, 16)
df.shape=(400, 13)
predictor_feat_df.shape=(400, 4)
predicted_feat_df.shape=(400, 4)
df_masked.shape=(400, 16)
d

##### Iterate over hyperparameters, fit models, analyze geometry on each:

In [292]:
# Iterate over dicts of hyperparamter combos:
pred_geo_results = pd.DataFrame()
pred_perf_results = pd.DataFrame()
pred_ae_results = pd.DataFrame()
start_mdl = time.time()
for hidx, curr_hparams in hparams_df.iterrows():

    # Get current model hypermarameter set:
    curr_autoencoder_params = dict(curr_hparams[autoencoder_cols])

    # Get current simulated whisker data:
    curr_whisker_sim = sim_df[sim_df.sim_idx==hidx]
    print(curr_whisker_sim.shape[0])
    
    # Fit model, test geometry:
    curr_results=mdl_geometry_pipeline(sim_params,  
        tasks=curr_hparams.task_defs, autoencoder_params=curr_autoencoder_params, xor=curr_hparams.xor, 
        n_geo_subsamples=curr_hparams.n_geo_subsamples, zscore_data=curr_hparams.zscore_data, 
        save_perf=False, sum_inpt=curr_hparams.sum_inpt, chunked_reconstruction_loss=curr_hparams.chunked_reconstruction_loss, 
        sessions_in=curr_whisker_sim, save_learning=curr_hparams.save_learning, gpu=curr_hparams.gpu, save_sessions=False, 
        verbose=True)

    curr_hparams_df = pd.DataFrame(hparams_df.iloc[0]).T

    # Extract geometry results, add metadata:
    curr_geo_results = curr_results['geo_df']
    geo_meta_cols = list(set(curr_hparams_df) - set(curr_geo_results.columns))
    geo_meta = pd.concat([curr_hparams_df[geo_meta_cols]]*curr_geo_results.shape[0],axis=0)
    geo_meta.index = np.arange(geo_meta.shape[0])
    curr_geo_results = pd.concat([curr_geo_results, geo_meta], axis=1)
    pred_geo_results = pd.concat([pred_geo_results, curr_geo_results], axis=0)
    
    # Extract classifier performance results, add metadata:
    curr_perf_results = curr_results['perf_df']
    perf_meta_cols = list(set(curr_hparams_df) - set(curr_perf_results.columns))
    perf_meta = pd.concat([curr_hparams_df[perf_meta_cols]]*curr_perf_results.shape[0],axis=0)
    perf_meta.index = np.arange(perf_meta.shape[0])
    curr_perf_results = pd.concat([curr_perf_results, perf_meta], axis=1)
    pred_perf_results = pd.concat([pred_perf_results, curr_perf_results], axis=0)

    # Extract autoencoder representations, add metadata:
    if curr_results['ae_df'] is not None:            
        curr_ae_results = curr_results['ae_df']
        ae_meta_cols = list(set(curr_hparams_df) - set(curr_ae_results.columns))
        ae_meta = pd.concat([curr_hparams_df[ae_meta_cols]]*curr_ae_results.shape[0],axis=0)
        curr_ae_results = pd.concat([curr_ae_results, perf_meta], axis=1)
        pred_ae_results = pd.concat([pred_ae_results, curr_ae_results])

pred_geo_results['train_partition'] = pred_geo_results.apply(lambda x :str(x.train_partition), axis=1)
stop_mdl = time.time()

400
Fitting autoencoder...
0 rec  120.90897369384766 ce  0.0 sp  2.6059789657592773 total  38234.77464259407
49 rec  1.4165854454040527 ce  0.0 sp  0.17357848584651947 total  447.963650772091
fit_autoencoder duration=1.758422613143921
400
Fitting autoencoder...
0 rec  111.19218444824219 ce  0.0 sp  2.9729483127593994 total  35162.05608659982
49 rec  1.5081998109817505 ce  0.0 sp  1.4745041131973267 total  476.93465693377624
fit_autoencoder duration=1.8128025531768799
400
Fitting autoencoder...
0 rec  35.53422546386719 ce  0.0 sp  1.2210859060287476 total  11236.908735577359
49 rec  0.9702388644218445 ce  0.0 sp  0.34935352206230164 total  306.8164685988336
fit_autoencoder duration=1.804091215133667
400
Fitting autoencoder...
0 rec  167.38589477539062 ce  0.0 sp  3.5060107707977295 total  52932.06756755128
49 rec  1.855696201324463 ce  0.0 sp  1.4007585048675537 total  586.8226641507673
fit_autoencoder duration=1.8146276473999023
400
Fitting autoencoder...
0 rec  101.13043212890625 ce  

##### Conslidate entangler and prediction model results:

In [293]:
perf_df = pd.concat([tngl_results['perf_df'], pred_perf_results], axis=0)
geo_df = pd.concat([tngl_results['geo_df'], pred_geo_results], axis=0)
ae_df = pd.concat([tngl_results['ae_df'], pred_ae_results], axis=0)

all_results = dict()
all_results['perf_df'] = perf_df
all_results['geo_df'] = geo_df
all_results['ae_df'] = ae_df

##### Save output:

In [294]:
if sv:
    
    # Save results dataframe:
    curr_output_directory=increment_dir_name(base_output_directory, run_base_name)
    if not os.path.exists(curr_output_directory):
        pathlib.Path(curr_output_directory).mkdir(parents=True, exist_ok=True)
    results_path = os.path.join(curr_output_directory, 'ae_iterate_beta_reconstruction.pickle')
    pickle.dump(all_results, open(results_path, 'wb'))
    
    M = Metadata()
    metadata_consts = find_df_constants(hparams_df)

    # Write task definitions:
    if 'task_defs' in metadata_consts:
        for t, task in enumerate(metadata_consts['task_defs']):
            curr_task_str = ' vs '.join([class_def2str(x) for x in task])
            M.add_param('task{}'.format(t), curr_task_str)

    # Write simulation parameters:
    sim_params = dict()
    for s in simulation_cols:
        if s in metadata_consts:
            sim_params[s] = metadata_consts[s]
    M.add_param('sim_params', sim_params)

    # Write entangler model parameters:
    M.add_param('tngl_model_params', tngl_mdl_params)
    
    # Write prediction model parameters:
    prediction_model_params = dict()
    for a in autoencoder_cols:
        if a in metadata_consts:
            prediction_model_params[a] = metadata_consts[a]
    if 'n_offsets' in prediction_model_params and prediction_model_params['n_offsets'] is None:
        prediction_model_params['n_offsets'] = 'auto'
    M.add_param('prediction_model_params', prediction_model_params)

    M.add_output(results_path)
    M.duration = stop_mdl - start_mdl
    metadata_path = os.path.join(curr_output_directory, 'ae_iterate_hidden_size_metadata.json')
    write_metadata(M, metadata_path)

Computing checksum for /mnt/smb/locker/issa-locker/users/Dan/code/ws/results/run759/ae_iterate_beta_reconstruction.pickle...
